In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis
import warnings

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

In [ ]:
product_type_1 = pd.read_csv('../../data_processed/product_type_1.csv', header=[0, 1])
product_type_2 = pd.read_csv('../../data_processed/train_2.csv', header=[0, 1])

print("="*60)
print("type1,2 데이터 로드 완료!")
print("="*60)

In [ ]:
# 결측치 확인
print("\n" + "="*60)
print("결측치 확인")
print("="*60)

missing_df = pd.DataFrame({
    '결측수': product_type_1.isnull().sum(),
    '결측비율(%)': (product_type_1.isnull().sum() / len(product_type_1) * 100).round(2)
})
missing_df = missing_df[missing_df['결측수'] > 0].sort_values('결측수', ascending=False)

if len(missing_df) > 0:
    print("\n[결측치 현황]")
    display(missing_df)
else:
    print("\n결측치 없음")

In [ ]:
# 결측치 중앙값 대체
na_cols = product_type_1.columns[product_type_1.isnull().any()]
product_type_1[na_cols] = product_type_1[na_cols].fillna(product_type_1[na_cols].median())

print("="*60)
print("중앙값 대체 완료!!")
print("="*60)
print(f"결측치 데이터 수: {product_type_1.isnull().any().sum()}개")
print("="*60)

In [ ]:
# 중복 shot 값 확인
shots = product_type_1['process', 'shot']
vc = shots.value_counts().sort_values(ascending=False)
shot_df = vc.to_frame("count")
dup_shot_count = shots.value_counts()
print("중복(2회 이상) shot 수:", (dup_shot_count >= 2).sum())
print("전체 shot 수:", len(dup_shot_count))
display(shot_df)

# 애초에 train 데이터는 샷이 없음..

In [ ]:
# 중복된 shot 번호를 가진 모든 행 추출 (원본 포함)
dup_all = product_type_1[product_type_1[('process', 'shot')].duplicated(keep=False)]

# shot 번호 기준으로 정렬해서 같은 shot끼리 붙여서 보기
display(dup_all.sort_values(('process', 'shot')))

#     cavity1  
# 샷   ↗         ➡️ 샷 번호가 같음
#     ↘
#     cavity2

# 1. Shot 번호는 단순 순번 (식별자)
# 2. 같은 Shot 번호여도 공정 값이 다름 → cavity 차이
# 3. cavity 정보는 이미 공정 컬럼들에 반영됨
# 4. Shot 번호와 불량 발생 사이의 인과관계가 없어 보임

# 중복된 shot 번호를 가진 행들만 추출
dup_shots = product_type_1[product_type_1.duplicated(subset=[('process', 'shot')], keep=False)].copy()

# shot별 최대 불량 여부 집계 (튜플 컬럼은 리스트로 감싸야 함)
shot_defect = dup_shots.groupby(('process', 'shot'))[[('defect_flag', 'is_defect')]].max()
shot_defect.columns = ['is_defect']
shot_defect = shot_defect.reset_index()
shot_defect.columns = ['shot', 'is_defect']

# shot 구간별 불량률 계산
shot_defect['shot_bin'] = pd.cut(shot_defect['shot'], bins=10)
defect_rate = shot_defect.groupby('shot_bin', observed=True)['is_defect'].mean()

# 시각화
defect_rate.plot(kind='bar', figsize=(12, 5), title='shot 구간별 불량률 (중복 shot 기준)')
plt.ylabel('불량률')
plt.xlabel('shot 구간')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# shot 번호가 커질수록 불량이 줄거나 늘어나는 패턴이 없음

In [ ]:
# 1. 한 샷 당 캐비티 별 불량률
defect_cols = product_type_1['defects'].columns
# _1로 끝나는 불량 유형명 추출 → 기준으로 _1/_2 짝 맞추기
defect_types = [col[:-2] for col in defect_cols if col.endswith("_1")]
rows = []
for col in defect_types:
    col1_key = ('defects', f'{col}_1')
    col2_key = ('defects', f'{col}_2')
    cavity1_rate = product_type_1[col1_key].mean() if col1_key in product_type_1.columns else None
    cavity2_rate = product_type_1[col2_key].mean() if col2_key in product_type_1.columns else None
    rows.append({
        'defect_type': col,
        'cavity1': cavity1_rate,
        'cavity2': cavity2_rate
    })

df_cavity = pd.DataFrame(rows)
print(df_cavity)


In [ ]:
# cavity1, 2 유형 별 양품 비율
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# cavity별 전체 불량률 (각 유형 불량률 합산)
c1_defect_rate = df_cavity['cavity1'].sum()
c2_defect_rate = df_cavity['cavity2'].sum()

# 정상품 비율 = 1 - 불량률
c1_normal_rate = 1 - c1_defect_rate
c2_normal_rate = 1 - c2_defect_rate

# --- 파이차트: cavity1 정상 vs 불량 비율 ---
axes[0, 0].pie(
    [c1_normal_rate, c1_defect_rate],
    labels=['정상', '불량'],
    autopct='%1.1f%%',
    startangle=90,
    colors=['steelblue', 'tomato']
)
axes[0, 0].set_title('Cavity1 정상 vs 불량 비율')

# --- 파이차트: cavity2 정상 vs 불량 비율 ---
axes[0, 1].pie(
    [c2_normal_rate, c2_defect_rate],
    labels=['정상', '불량'],
    autopct='%1.1f%%',
    startangle=90,
    colors=['steelblue', 'tomato']
)
axes[0, 1].set_title('Cavity2 정상 vs 불량 비율')

# --- 바차트: 불량 유형별 cavity1 vs cavity2 정상품 비율 ---
x = range(len(df_cavity))
width = 0.35

c1_normal_by_type = 1 - df_cavity['cavity1'].fillna(0)
c2_normal_by_type = 1 - df_cavity['cavity2'].fillna(0)

axes[1, 0].bar([i - width/2 for i in x], c1_normal_by_type, width=width, label='cavity1', color='steelblue')
axes[1, 0].bar([i + width/2 for i in x], c2_normal_by_type, width=width, label='cavity2', color='tomato')
axes[1, 0].set_xticks(list(x))
axes[1, 0].set_xticklabels(df_cavity['defect_type'], rotation=30, ha='right')
axes[1, 0].set_ylabel('정상품 비율')
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_title('불량 유형별 Cavity1 vs Cavity2 정상품 비율')
axes[1, 0].legend()

# --- 바차트: cavity별 전체 정상품 비율 ---
normal_totals = [c1_normal_rate, c2_normal_rate]
bars = axes[1, 1].bar(['cavity1', 'cavity2'], normal_totals, color=['steelblue', 'tomato'])
for bar, val in zip(bars, normal_totals):
    axes[1, 1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                    f'{val:.4f}', ha='center', va='bottom')
axes[1, 1].set_ylabel('정상품 비율')
axes[1, 1].set_ylim(0, 1.05)
axes[1, 1].set_title('Cavity 타입별 전체 정상품 비율')

plt.tight_layout()
plt.show()


In [ ]:
# cavity1, 2 유형 별 양품 비율
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 불량 유형별 고정 색상 매핑
all_defect_types = df_cavity['defect_type'].tolist()
color_palette = plt.cm.tab10.colors
color_map = {dtype: color_palette[i % len(color_palette)] for i, dtype in enumerate(all_defect_types)}

# --- 파이차트: cavity1 불량 유형별 비율 ---
c1 = df_cavity.dropna(subset=['cavity1'])
axes[0, 0].pie(
    c1['cavity1'],
    labels=c1['defect_type'],
    autopct='%1.1f%%',
    startangle=90,
    colors=[color_map[t] for t in c1['defect_type']]
)
axes[0, 0].set_title('Cavity1 불량 유형별 비율')

# --- 파이차트: cavity2 불량 유형별 비율 ---
c2 = df_cavity.dropna(subset=['cavity2'])
axes[0, 1].pie(
    c2['cavity2'],
    labels=c2['defect_type'],
    autopct='%1.1f%%',
    startangle=90,
    colors=[color_map[t] for t in c2['defect_type']]
)
axes[0, 1].set_title('Cavity2 불량 유형별 비율')

# --- 바차트: 불량 유형별 cavity1 vs cavity2 불량률 비교 ---
x = range(len(df_cavity))
width = 0.35

bars1 = axes[1, 0].bar([i - width/2 for i in x], df_cavity['cavity1'].fillna(0), width=width, label='cavity1')
bars2 = axes[1, 0].bar([i + width/2 for i in x], df_cavity['cavity2'].fillna(0), width=width, label='cavity2')
axes[1, 0].set_xticks(list(x))
axes[1, 0].set_xticklabels(df_cavity['defect_type'], rotation=30, ha='right')
axes[1, 0].set_ylabel('불량률')
axes[1, 0].set_title('불량 유형별 Cavity1 vs Cavity2 불량률')
axes[1, 0].legend()

# --- 바차트: cavity별 전체 불량률 합계 ---
cavity_totals = [df_cavity['cavity1'].sum(), df_cavity['cavity2'].sum()]
bars = axes[1, 1].bar(['cavity1', 'cavity2'], cavity_totals, color=['steelblue', 'tomato'])
for bar, val in zip(bars, cavity_totals):
    axes[1, 1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                    f'{val:.4f}', ha='center', va='bottom')
axes[1, 1].set_ylabel('불량률 합계')
axes[1, 1].set_title('Cavity 타입별 전체 불량률 합계')

plt.tight_layout()
plt.show()

# cavity1 에서 더 많은 불량률과 불량유형이 보임


In [ ]:
# 컬럼을 하나로 통합 했을 때 불량률 및 불량유형 별 불량률


In [ ]:
# cavity1과 2의 불량 유형이 확연히 다르다.
# cavity1의 불량유형 종류와 불량률이 cavity2 보다 높다.
# 미성형(short_shot): 캐비티 내 충진이 완벽히 되지 않아 제품이 완성되지 않는 것
# 박리(exfoliation): 제품의 표면 등이 벗겨지는 현상. 충진 응고 부족, 부풀음, 칠층 등이 요인
# 기공(Blow Hole): 고체 재료 속에 기포가 들어가 생긴 중공의 구멍으로써 주물 내부에 생기는 결함의 한 종류
# 평탄(Deformation): 금속은 응고되면 수축이 생기고 이 수축량이 재료의 각 부분에 서로 다를 때 내부 잔류 응력이 발생하여 변형이 생기는 결함
# 개재물(Inclusions): 용탕 속에 섞이는 각종 불순물로 주형의 청소상태가 불량하거나 용탕, 용해 찌꺼기가 있을 때의 결함

# 불량 유형별 관련 변수

# Short Shot (충진 미완료)
# process: casting_pressure, cylinder_pressure, velocity_1, velocity_2, velocity_3, high_velocity, rapid_rise_time
# sensor:  melting_furnace_temp

# Deformation (제품 변형)
# process: casting_pressure, clamping_force, cycle_time, spray_time, spray_1_time, spray_2_time
# sensor:  coolant_temp, coolant_temp_min, coolant_temp_max, coolant_pressure

# Bubble (내부 기포)
# process: velocity_1, velocity_2, velocity_3, high_velocity, casting_pressure, pressure_rise_time
# sensor:  air_pressure, air_pressure_min, air_pressure_max

# Exfoliation (박리)
# process: spray_time, spray_1_time, spray_2_time, casting_pressure
# sensor:  coolant_temp, coolant_temp_min, coolant_temp_max, factory_temp

# 공통 주요 변수
# casting_pressure  → 4개 불량 유형 모두 관련
# velocity 계열     → Short Shot, Bubble 공통
# coolant_temp 계열 → Deformation, Exfoliation 공통


In [ ]:
# 공정, 센서 별 불량유형: 정상 vs 불량 공정값 분포 비교 (boxplot)
defect_vars = {
    'short_shot': {
        'process': ['casting_pressure', 'cylinder_pressure', 'velocity_1', 'velocity_2', 'velocity_3', 'high_velocity', 'rapid_rise_time', 'biscuit_thickness', 'clamping_force', 'cycle_time', 'pressure_rise_time', 'casting_pressure', 'spray_time', 'spray_1_time', 'spray_2_time' ],
        'sensor':  ['melting_furnace_temp']
    },
    'deformation': {
        'process': ['casting_pressure', 'clamping_force', 'cycle_time', 'spray_time', 'spray_1_time', 'spray_2_time'],
        'sensor':  ['coolant_temp', 'coolant_temp_min', 'coolant_temp_max', 'coolant_pressure']
    },
    'bubble': {
        'process': ['velocity_1', 'velocity_2', 'velocity_3', 'high_velocity', 'casting_pressure', 'pressure_rise_time'],
        'sensor':  ['air_pressure', 'air_pressure_min', 'air_pressure_max']
    },
    'exfoliation': {
        'process': ['spray_time', 'spray_1_time', 'spray_2_time', 'casting_pressure'],
        'sensor':  ['coolant_temp', 'coolant_temp_min', 'coolant_temp_max', 'factory_temp']
    },
    'blow_hole': {
        'process': ['velocity_1', 'velocity_2', 'velocity_3', 'high_velocity', 'pressure_rise_time'],
        'sensor':  ['air_pressure', 'air_pressure_min', 'air_pressure_max']
    },
    'stain': {
        'process': ['spray_time', 'spray_1_time', 'spray_2_time'],
        'sensor':  ['coolant_temp', 'coolant_pressure', 'factory_humidity']
    },
    'contamination': {
        'process': ['spray_time', 'spray_1_time', 'spray_2_time'],
        'sensor':  ['factory_temp', 'factory_humidity', 'air_pressure']
    },
    'impurity': {
        'process': ['casting_pressure', 'cylinder_pressure'],
        'sensor':  ['melting_furnace_temp', 'air_pressure']
    }
}

defect_all_cols = product_type_2['defects'].columns.tolist()

for defect, vars_dict in defect_vars.items():
    all_vars = vars_dict['process'] + vars_dict['sensor']
    n_vars = len(all_vars)

    # _1, _2 접미사가 붙은 컬럼 찾기 → 어느 cavity라도 불량이면 1
    matched = [c for c in defect_all_cols if c.startswith(defect)]
    if not matched:
        print(f"[SKIP] '{defect}' 관련 defects 컬럼 없음")
        continue
    defect_series = product_type_2[('defects', matched[0])].copy()
    for m in matched[1:]:
        defect_series = defect_series | product_type_2[('defects', m)]

    inner_fig, inner_axes = plt.subplots(1, n_vars, figsize=(n_vars * 4, 5), squeeze=False)
    inner_fig.suptitle(f'{defect}: 정상 vs 불량 공정값 분포 비교', fontsize=14)

    for j, var in enumerate(all_vars):
        col_type = 'process' if var in vars_dict['process'] else 'sensor'

        if (col_type, var) not in product_type_2.columns:
            inner_axes[0, j].set_title(f'{var}\n(컬럼 없음)')
            inner_axes[0, j].axis('off')
            continue

        plot_df = pd.DataFrame({
            'value': product_type_2[(col_type, var)],
            'defect': defect_series.astype(str)
        })

        sns.boxplot(x='defect', y='value', data=plot_df, ax=inner_axes[0, j],
                    palette={'0': 'steelblue', '1': 'tomato'},
                    order=['0', '1'])
        inner_axes[0, j].set_title(var)
        inner_axes[0, j].set_xticklabels(['정상', '불량'])
        inner_axes[0, j].set_xlabel('')
        inner_axes[0, j].set_ylabel('값')

    plt.tight_layout()
    plt.show()


In [ ]:
# 불량 유형별 공정/센서 변수의 정상 vs 불량 IQR 비교
iqr_rows = []

for defect, vars_dict in defect_vars.items():
    all_vars = vars_dict['process'] + vars_dict['sensor']

    matched = [c for c in defect_all_cols if c.startswith(defect)]
    if not matched:
        continue
    defect_series = product_type_1[('defects', matched[0])].copy()
    for m in matched[1:]:
        defect_series = defect_series | product_type_1[('defects', m)]

    for var in all_vars:
        col_type = 'process' if var in vars_dict['process'] else 'sensor'
        if (col_type, var) not in product_type_1.columns:
            continue

        values = product_type_1[(col_type, var)]
        normal_vals = values[defect_series == 0]
        defect_vals = values[defect_series == 1]

        iqr_rows.append({
            'defect_type': defect,
            'variable': var,
            'normal_Q1':  normal_vals.quantile(0.25),
            'normal_Q3':  normal_vals.quantile(0.75),
            'normal_IQR': normal_vals.quantile(0.75) - normal_vals.quantile(0.25),
            'defect_Q1':  defect_vals.quantile(0.25),
            'defect_Q3':  defect_vals.quantile(0.75),
            'defect_IQR': defect_vals.quantile(0.75) - defect_vals.quantile(0.25),
        })

df_iqr = pd.DataFrame(iqr_rows)
df_iqr = df_iqr.round(4)
display(df_iqr)


In [ ]:
# defect_vars 공정/센서 변수 이상치 확인 (IQR 방법)
outlier_rows = []

for defect, vars_dict in defect_vars.items():
    all_vars = vars_dict['process'] + vars_dict['sensor']

    for var in all_vars:
        col_type = 'process' if var in vars_dict['process'] else 'sensor'
        if (col_type, var) not in product_type_1.columns:
            continue

        values = product_type_1[(col_type, var)]
        Q1 = values.quantile(0.25)
        Q3 = values.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers = values[(values < lower) | (values > upper)]

        outlier_rows.append({
            'defect_type': defect,
            'variable': var,
            'Q1': Q1,
            'Q3': Q3,
            'IQR': IQR,
            'lower_bound': lower,
            'upper_bound': upper,
            'outlier_count': len(outliers),
            'outlier_ratio(%)': round(len(outliers) / len(values) * 100, 2)
        })

df_outlier = pd.DataFrame(outlier_rows).round(4)
df_outlier = df_outlier.sort_values('outlier_ratio(%)', ascending=False)
display(df_outlier)


In [ ]:
# X_train_1 기반 다중공선성 확인 (VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_train_1 = pd.read_csv('../../data_processed/X_train_1.csv')

vif_input = X_train_1.dropna()

vif_result = pd.DataFrame({
    'variable': vif_input.columns,
    'VIF': [variance_inflation_factor(vif_input.values, i)
            for i in range(vif_input.shape[1])]
}).sort_values('VIF', ascending=False).round(2)

print("VIF 기준: 1=독립, 1~5=낮음, 5~10=주의, 10 이상=높은 다중공선성")
display(vif_result)


In [ ]:
corr = X_train_1.corr()

plt.figure(figsize=(15, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, center=0)
plt.title('공정/센서 변수 상관계수 heatmap')
plt.tight_layout()
plt.show()

In [ ]:
for defect, vars_dict in defect_vars.items():
    all_vars = vars_dict['process'] + vars_dict['sensor']
    n_vars = len(all_vars)

    matched = [c for c in defect_all_cols if c.startswith(defect)]
    if not matched:
        print(f"[SKIP] '{defect}' 관련 defects 컬럼 없음")
        continue
    defect_series = product_type_1[('defects', matched[0])].copy()
    for m in matched[1:]:
        defect_series = defect_series | product_type_1[('defects', m)]

    # 언더샘플링
    defect_idx = defect_series[defect_series == 1].index
    normal_idx = defect_series[defect_series == 0].sample(n=len(defect_idx), random_state=42).index
    balanced_idx = defect_idx.union(normal_idx)

    balanced_data = product_type_1.loc[balanced_idx]
    balanced_defect = defect_series.loc[balanced_idx]

    inner_fig, inner_axes = plt.subplots(1, n_vars, figsize=(n_vars * 4, 5), squeeze=False)
    inner_fig.suptitle(f'{defect}: 정상 vs 불량 공정값 분포 비교 (언더샘플링)', fontsize=14)

    for j, var in enumerate(all_vars):
        col_type = 'process' if var in vars_dict['process'] else 'sensor'

        if (col_type, var) not in product_type_1.columns:
            inner_axes[0, j].set_title(f'{var}\n(컬럼 없음)')
            inner_axes[0, j].axis('off')
            continue

        plot_df = pd.DataFrame({
            'value': balanced_data[(col_type, var)],
            'defect': balanced_defect.astype(str)
        })

        sns.boxplot(x='defect', y='value', data=plot_df, ax=inner_axes[0, j],
                    palette={'0': 'steelblue', '1': 'tomato'},
                    order=['0', '1'])
        inner_axes[0, j].set_title(var)
        inner_axes[0, j].set_xticklabels(['정상', '불량'])
        inner_axes[0, j].set_xlabel('')
        inner_axes[0, j].set_ylabel('값')

    plt.tight_layout()
    plt.show()

In [ ]:
# 불량 유형을 행 레이블로 갖는 DataFrame 생성
# 불량이 발생한 행만 추출 + defect_type 컬럼 추가

process_cols = [c for c in product_type_1['process'].columns if c != 'shot']
sensor_cols  = product_type_1['sensor'].columns.tolist()
defect_cols  = product_type_1['defects'].columns.tolist()

# _1/_2 묶어서 불량 유형 목록 추출
defect_types = list(dict.fromkeys(c.rsplit('_', 1)[0] for c in defect_cols))

# 공정/센서 컬럼만 추린 base DataFrame
base_cols = [('process', c) for c in process_cols] + [('sensor', c) for c in sensor_cols]
base_df = product_type_1[base_cols].copy()
base_df.columns = [f'{g}_{c}' for g, c in base_df.columns]  # 플랫 컬럼명으로 변환

rows = []
for dtype in defect_types:
    matched = [c for c in defect_cols if c.startswith(dtype)]
    is_defect = product_type_1[('defects', matched[0])].copy()
    for m in matched[1:]:
        is_defect = is_defect | product_type_1[('defects', m)]

    defect_rows = base_df[is_defect == 1].copy()
    defect_rows.insert(0, 'defect_type', dtype)
    rows.append(defect_rows)

df_defect_long = pd.concat(rows, ignore_index=True)

print(f'shape: {df_defect_long.shape}')
print(f'불량 유형별 건수:\n{df_defect_long["defect_type"].value_counts()}')
display(df_defect_long.head(10))